### Chaos Induction Engine

This notebook deliberately induces failure modes in the otherwise "Ideal" baseline data.
The goal is to introduce realism and messiness into the data as observed in Quick-commerce supply chain pipelines.

The failures are not random, each failure points to a physical cause.  

Read full fault specification: [`chaos_library.md`](chaos_library.md)



In [18]:
import yaml
import random
import numpy as np
import pandas as pd
from datetime import date, datetime, timedelta

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

random.seed(cfg["simulation"]["random_seed"])
np.random.seed(cfg["simulation"]["random_seed"])

# baseline events
df_events = pd.read_csv("events_baseline.csv", parse_dates=["event_timestamp"])

## dimension tables are needed for targeted chaos
df_scanner = pd.read_csv("dim_scanner.csv")
df_product = pd.read_csv("dim_product.csv")
df_node = pd.read_csv("dim_node.csv")
df_supplier = pd.read_csv("dim_supplier.csv")

## pre-build targeting sets (used across multiple chaos methods)
dead_rtc_scanners = set(df_scanner[df_scanner["battery_backed_rtc"] == False]["device_id"])
degraded_scanners = set(df_scanner[df_scanner["is_degraded"] == True]["device_id"])
old_firmware_devices = set(df_scanner[df_scanner["firmware_version"] == "v1.1.0"]["device_id"])
slow_scanners = set(df_scanner[df_scanner["avg_sync_latency_ms"] > 500]["device_id"])
high_failure_devices = set(df_scanner[df_scanner["failure_count_30d"] > 5]["device_id"])

email_suppliers = set(df_supplier[df_supplier["edi_format"] == "email_manual"]["supplier_id"])
xml_suppliers = set(df_supplier[df_supplier["edi_format"] == "SFTP_XML"]["supplier_id"])
csv_suppliers = set(df_supplier[df_supplier["edi_format"] == "SFTP_CSV"]["supplier_id"])

email_products = set(df_product[df_product["supplier_id"].isin(email_suppliers)]["product_id"])
xml_products = set(df_product[df_product["supplier_id"].isin(xml_suppliers)]["product_id"])
csv_products = set(df_product[df_product["supplier_id"].isin(csv_suppliers)]["product_id"])

print(f"Baseline events loaded: {len(df_events):,}")
print(f"dead RTC scanners: {len(dead_rtc_scanners)}")
print(f"Degraded scanners: {len(degraded_scanners)}")
print(f"Old firmware devices: {len(old_firmware_devices)}")
print(f"email_manual products: {len(email_products)}")
print(f"SFTP_XML products: {len(xml_products)}")

Baseline events loaded: 436,143
dead RTC scanners: 47
Degraded scanners: 66
Old firmware devices: 103
email_manual products: 28
SFTP_XML products: 21


In [19]:
class SupplyChainChaosEngine:
    """ 
    Injects 25 physically attributed failure modes into the clean baseline.
    Every chaos method targets specific rows based on scanner properties, supplier
    EDI format, or operational conditions, never uniform sampling. 
    """

    def __init__(self, df_events, df_node):
        self.df = df_events.copy()
        self.df_nodes = df_node.copy()
        self.df["event_timestamp"] = pd.to_datetime(self.df["event_timestamp"])
        self.injection_log = [] # tracks what was injected for summary

    def _log(self, chaos_id, name, rows_affected):
        """Records each injection for post run summary"""
        self.injection_log.append({
            "chaos_id": chaos_id,
            "name": name,
            "rows_affected": rows_affected,
        })
        print(f" [{chaos_id:02d}] {name:<40} {rows_affected:>6} rows")

    def save(self, events_path="events_chaotic.csv", nodes_path="nodes_chaotic.csv"):
        import os
        os.makedirs("data", exist_ok=True)
        self.df.to_csv(events_path, index=False)
        self.df_nodes.to_csv(nodes_path, index=False)
        print(f"\nSaved: {events_path} ({len(self.df):,} rows)")
        print(f"Saved: {nodes_path} ({len(self.df_nodes)} rows)")

    def summary(self):
        print("\n === Chaos Injection Summary ===")
        total = 0
        for entry in self.injection_log:
            print(f" [{entry['chaos_id']:02d}] {entry['name']:<40} {entry['rows_affected']:>6}")
            total += entry["rows_affected"]
        print(f"\n Total rows touched: {total:,}")
        print(f" Total events in chaotic dataset: {len(self.df):,}")

### **Apply Temporal Chaos:**


In [20]:
def apply_temporal_chaos(self):
    """ 
    Injects 6 temporal failures.
    All target specific scanner properties from dim_scanner.
    See chaos_library.md for full specifications.
    """
    print("Applying temporal chaos...")

    # -- 1.1 UTC/IST Drift
    # scanners with dead rtc battery reset to utc on power loss.
    # all their events are 5h 30min behind real time
    mask = self.df["device_id"].isin(dead_rtc_scanners)
    affected = mask.sum()
    self.df.loc[mask, "event_timestamp"] -= pd.Timedelta(hours=5, minutes=30)
    self._log(1, "UTC/IST Drift", affected)

    # -- 1.2 Timezone double-conversion
    # v1.1.0 firmware: utc to ist applied twice so 11h ahead
    # applied to 2% of old firmware (different rows from drift) 
    old_fw_mask = self.df["device_id"].isin(old_firmware_devices)
    old_fw_idx = self.df[old_fw_mask & ~mask].sample(
        frac=0.02, random_state=42
    ).index
    self.df.loc[old_fw_idx, "event_timestamp"] += pd.Timedelta(hours=11)
    self._log(2, "Timezone Double-Conversion", len(old_fw_idx))

    # -- 1.3 Batch Heartbeat 
    # high latency scanners upload all the data buffered events at 23:59:59
    # pick 3 random dates, force all slow scanner events to EOD
    batch_dates = pd.to_datetime(
        pd.Series(self.df["event_timestamp"].dt.date.unique())
        .sample(3, random_state=42).values
    )
    affected_bh = 0
    for batch_date in batch_dates:
        bh_mask = (
            self.df["device_id"].isin(slow_scanners) & 
            (self.df["event_timestamp"].dt.date == batch_date.date())
        )
        affected_bh += bh_mask.sum()
        self.df.loc[bh_mask, "event_timestamp"] = pd.Timestamp(
            f"{batch_date.date()} 23:59:59"
        )
    self._log(3, "Batch Heartbeat (3 dates)", affected_bh)

    # -- 1.4 Future Timestamp
    # v1.1.0 firmware clock drifts forward after battery replacement
    # 2% of old firmware rows shifted 2-45 days into the future
    future_idx = self.df[old_fw_mask].sample(frac=0.02, random_state=43).index
    self.df.loc[future_idx, "event_timestamp"] = (
        self.df.loc[future_idx, "event_timestamp"]
        + pd.to_timedelta(
            [random.randint(2, 45) for _ in range(len(future_idx))],
            unit="D"
        )
    )
    self._log(4, "Future Timestamp", len(future_idx))

    # -- 1.5 Midnight Rollover
    # old ERP systems truncate datetime to date only during export
    # 5% of v1.1.0 rows get time component zeroed    
    midnight_idx = self.df[old_fw_mask].sample(frac=0.05, random_state=44).index
    self.df.loc[midnight_idx, "event_timestamp"] = (
        self.df.loc[midnight_idx, "event_timestamp"].dt.normalize()
    )
    self._log(5, "Midnight Rollover", len(midnight_idx))    

    # -- 1.6 Processing Lag Smear
    ## degraded scanners accumulate massive processing delays
    # 2% of degraded scanner rows get lag set to 155.5 hours
    lag_idx = self.df[
        self.df["device_id"].isin(degraded_scanners)
    ].sample(frac=0.02, random_state=45).index

    self.df.loc[lag_idx, "processing_lag_hours"] = 155.5
    self._log(6, "Processing Lag Smear", len(lag_idx))

    return self   

# patch the method on the class
SupplyChainChaosEngine.apply_temporal_chaos = apply_temporal_chaos
print("apply_temporal_chaos() patched onto ChaosEngine.")

apply_temporal_chaos() patched onto ChaosEngine.


### **Apply Structural Chaos:**

In [21]:
def apply_structural_chaos(self):
    """
    Injects 5 structural failures.
    Targets email_manual supplier products, SFTP_XML products,
    and nodes with high-failure scanners.
    See chaos_library.md Category 2 for full specifications.
    """
    print("Applying structural chaos...")

    # -- 2.1 Schema Poisoning — Unit Suffix 
    # email_manual suppliers have human data entry
    # "130 units" instead of 130 causes problems
    email_rows = self.df[
        self.df["product_id"].isin(email_products) &
        (self.df["units_sold"] > 0)
    ]
    poison_idx = email_rows.sample(
        n=min(50, len(email_rows)), random_state=46
    ).index
    self.df["units_sold"] = self.df["units_sold"].astype(object)
    self.df.loc[poison_idx, "units_sold"] = (
        self.df.loc[poison_idx, "units_sold"].astype(str) + " units"
    )
    self._log(7, "Schema Poisoning — Unit Suffix", len(poison_idx))

    # -- 2.2 Schema Poisoning — Decimal Separator
    # SFTP_XML suppliers used comma instead of decimal separator
    # "249,99" instead of 249.99 — looks like integer to Indian systems
    units_num = pd.to_numeric(self.df["units_sold"].astype(str).str.replace(" units", ""), errors="coerce")
    xml_rows = self.df[
        self.df["product_id"].isin(xml_products) &
        (units_num > 0)
    ]
    decimal_idx = xml_rows.sample(
        frac=0.03, random_state=47
    ).index
    # inject into processing_lag_hours as a proxy numeric field
    # (units_sold is already poisoned above for some rows)
    self.df["processing_lag_hours"] = self.df["processing_lag_hours"].astype(object)
    self.df.loc[decimal_idx, "processing_lag_hours"] = (
        self.df.loc[decimal_idx, "processing_lag_hours"]
        .apply(lambda x: str(x).replace(".", ","))
    )
    self._log(8, "Schema Poisoning: Decimal Separator", len(decimal_idx))

    # -- 2.3 Address and City Corruption
    # cracked scanner touchscreen may introduce emoji and special characters
    # targets nodes whose scanners have high failure counts
    bad_chars = [
        "🚚 Hub-Alpha",
        "Sector 12 || 🏠",
        "Industrial Area\tPhase-1",
        "AREA-999-###",
    ]
    high_failure_nodes = set(
        df_scanner[df_scanner["failure_count_30d"] > 5]["assigned_node_id"]
    )
    corrupt_node_idx = self.df_nodes[
        self.df_nodes["node_id"].isin(high_failure_nodes)
    ].sample(n=min(3, len(high_failure_nodes)), random_state=48).index

    self.df_nodes.loc[corrupt_node_idx, "city"] = [
        random.choice(bad_chars) for _ in range(len(corrupt_node_idx))
    ]
    self._log(9, "Address and City Corruption", len(corrupt_node_idx))

    # -- 2.4 Null RTO Reason 
    # returns scanner staff skip reason dropdown under pressure
    # units_rto > 0 but rto_reason is NULL (referential integrity violation)
    rto_rows = self.df[
        (self.df["event_type"] == "rto_return") &
        (self.df["units_rto"] > 0) &
        (self.df["device_id"].isin(
            df_scanner[df_scanner["scanner_zone"] == "returns"]["device_id"]
        ))
    ]
    null_rto_idx = rto_rows.sample(
        n=min(20, len(rto_rows)), random_state=49
    ).index
    self.df.loc[null_rto_idx, "rto_reason"] = None
    self._log(10, "Null RTO Reason", len(null_rto_idx))

    # -- 2.5 UoM Mismatch: Cases vs Units 
    # email_manual supplier sends quantities in cases (24 units each)
    # WMS expects individual units, 100 cases looks like 100 units
    transfer_rows = self.df[
        (self.df["event_type"] == "inbound_transfer") &
        (self.df["product_id"].isin(email_products))
    ]
    uom_idx = transfer_rows.sample(
        n=min(5, len(transfer_rows)), random_state=50
    ).index
    self.df.loc[uom_idx, "units_transferred"] = (
        self.df.loc[uom_idx, "units_transferred"] // 24
    ).clip(lower=1)
    self._log(11, "UoM Mismatch — Cases vs Units", len(uom_idx))

    return self

# patch the method on the class
SupplyChainChaosEngine.apply_structural_chaos = apply_structural_chaos
print("apply_structural_chaos() patched onto ChaosEngine.")

apply_structural_chaos() patched onto ChaosEngine.


### **Apply Operational Chaos:**

In [22]:
def apply_operational_chaos(self):
    """
    Injects 6 operational failures.
    These require cross-row logic: individual fields look valid
    but their combination is impossible or contradictory.
    See chaos_library.md Category 3 for full specifications.
    """
    print("Applying operational chaos...")

    # -- 3.1 Ghost Inventory 
    # sale forced before inbound scan confirms stock arrival
    # units_sold exceeds stock_on_hand which is physically impossible
    safe_numeric_units = pd.to_numeric(
        self.df["units_sold"].astype(str).str.replace(" units", "", case=False, regex=False), 
        errors="coerce"
    )
    fulfillment_rows = self.df[
        (self.df["event_type"] == "customer_fulfillment") &
        (safe_numeric_units > 0)
    ]

    ghost_idx = fulfillment_rows.sample(
        n=min(20, len(fulfillment_rows)), random_state=51
    ).index
    self.df["units_sold"] = self.df["units_sold"].astype(object)
    self.df.loc[ghost_idx, "units_sold"] = (
        self.df.loc[ghost_idx, "stock_on_hand"] + 
        pd.Series([random.randint(50, 500) for _ in range(len(ghost_idx))], index=ghost_idx)
    )
    self._log(12, "Ghost Inventory", len(ghost_idx))

    # -- 3.2 Duplicate Transaction: exact dupes
    # double tap submit on outbound scanner
    # exact copy of 100 rows (caught by simple deduplication)
    outbound_rows = self.df[
        (self.df["event_type"] == "customer_fulfillment") &
        (self.df["device_id"].isin(
            df_scanner[df_scanner["scanner_zone"] == "outbound"]["device_id"]
        ))
    ]
    exact_dupes = outbound_rows.sample(
        n=min(100, len(outbound_rows)), random_state=52
    )
    self.df = pd.concat([self.df, exact_dupes], ignore_index=True)
    self._log(13, "Duplicate Echoes — Exact", len(exact_dupes))

    # -- 3.3 Duplicate Transaction Echoes: Sub-second 
    # same event with +/-millisecond timestamp offset
    # evades simple df.duplicated() 
    near_dupes = outbound_rows.sample(
        n=min(50, len(outbound_rows)), random_state=53
    ).copy()
    ms_jitter = pd.to_timedelta(
        [random.randint(1, 999) for _ in range(len(near_dupes))],
        unit="ms"
    )
    near_dupes["event_timestamp"] = near_dupes["event_timestamp"] + ms_jitter
    self.df = pd.concat([self.df, near_dupes], ignore_index=True)
    self._log(14, "Duplicate Echoes — Sub-second", len(near_dupes))

    # -- 3.4 Node Blackout 
    # entire scanner network offline for 4 consecutive days
    # no events at all (not even snapshots) for one node
    blackout_node = self.df["node_id"].value_counts().index[5]  # pick a busy node
    blackout_days = [10, 11, 12, 13]   # simulation days 10-13
    from datetime import date as date_type
    sim_start = date_type(2025, 12, 1)
    blackout_dates = [
        sim_start + pd.Timedelta(days=d)
        for d in blackout_days
    ]
    before = len(self.df)
    self.df = self.df[~(
        (self.df["node_id"] == blackout_node) &
        (self.df["event_timestamp"].dt.date.isin(
            [d.date() if hasattr(d, 'date') else d for d in blackout_dates]
        ))
    )]
    deleted = before - len(self.df)
    self._log(15, f"Node Blackout ({blackout_node})", deleted)

    # -- 3.5 Reverse Logistics Void 
    # return logged but stock not physically restocked
    # units_rto = 100 but stock_on_hand stays at pre-return level
    rto_rows = self.df[
        (self.df["event_type"] == "rto_return") &
        (self.df["units_rto"] > 0)
    ]
    void_idx = rto_rows.sample(
        n=min(15, len(rto_rows)), random_state=54
    ).index
    self.df.loc[void_idx, "units_rto"] = 100
    self.df.loc[void_idx, "stock_on_hand"] = 2
    self._log(16, "Reverse Logistics Void", len(void_idx))

    # -- 3.6 Lateral Transfer Without Source Deduction 
    # DS2 receives stock but no outbound event exists for DS1
    # total system inventory exceeds physical reality
    transfer_rows = self.df[
    (self.df["event_type"] == "lateral_transfer") &
    (self.df["source_node"].notna()) &
    (self.df["source_node"].str.startswith("DS_", na=False))
    ]
    orphan_idx = transfer_rows.sample(
        n=min(5, len(transfer_rows)), random_state=55
    ).index
    self.df.loc[orphan_idx, "source_node"] = "MISSING_SOURCE"
    self._log(17, "Lateral Transfer Without Deduction", len(orphan_idx))

    return self

# patch the method on the class
SupplyChainChaosEngine.apply_operational_chaos = apply_operational_chaos
print("apply_operational_chaos() patched onto ChaosEngine.")

apply_operational_chaos() patched onto ChaosEngine.


### **Apply Market Chaos:**

In [23]:
def apply_market_chaos(self):
    """
    Injects 5 market and integrity failures.
    These corrupt the meaning of values over time rather than
    breaking individual fields. Slowest-moving and hardest to detect.
    See chaos_library.md Category 4 for full specifications.
    """
    print("Applying market chaos...")

    # -- 4.1 GST Math Drift 
    # SFTP_XML supplier ERP reconfigured to include 18% GST in unit price
    # revenue appears to grow when it is actually a pricing error
    xml_fulfillments = self.df[
        (self.df["event_type"] == "customer_fulfillment") &
        (self.df["product_id"].isin(xml_products))
    ]
    gst_idx = xml_fulfillments.sample(
        frac=0.15, random_state=56
    ).index
    self.df.loc[gst_idx, "units_sold"] = (
        self.df.loc[gst_idx, "units_sold"]
        .apply(lambda x: x if not str(x).replace(".", "").isdigit()
               else int(float(str(x).replace(" units", "")) * 1.18))
    )
    self._log(18, "GST Math Drift (+18%)", len(gst_idx))

    # -- 4.2 SKU Migration — Identity Crisis 
    # PROD_001 renamed mid-simulation after March 10th
    # historical analytics sees two products where one exists
    cutoff = pd.Timestamp("2026-03-10")
    sku_mask = (
        (self.df["product_id"] == "PROD_001") &
        (self.df["event_timestamp"] < cutoff)
    )
    self.df.loc[sku_mask, "product_id"] = "PROD_001_OLD_SKU"
    self._log(19, "SKU Migration — PROD_001", sku_mask.sum())

    # -- 4.3 Fat Finger Outlier 
    # cracked touchscreen registers digit 9 multiple times
    # 999999 units sold 10,000x normal daily volume
    fat_finger_rows = self.df[
        self.df["device_id"].isin(high_failure_devices) &
        (self.df["event_type"] == "customer_fulfillment")
    ]
    ff_idx = fat_finger_rows.sample(
        n=min(2, len(fat_finger_rows)), random_state=57
    ).index
    self.df.loc[ff_idx, "units_sold"] = 999999
    self._log(20, "Fat Finger Outlier (999999 units)", len(ff_idx))

    # -- 4.4 Price Drift — Supplier Rounding Error 
    # SFTP_CSV supplier system adds floating point noise to prices
    # ₹249.99 becomes ₹249.994318, this breaks invoice reconciliation
    csv_fulfillments = self.df[
        (self.df["event_type"] == "customer_fulfillment") &
        (self.df["product_id"].isin(csv_products))
    ]
    drift_idx = csv_fulfillments.sample(
        frac=0.08, random_state=58
    ).index
    noise = pd.Series(
        [random.uniform(-0.05, 0.05) for _ in range(len(drift_idx))],
        index=drift_idx
    )
    # inject into processing_lag_hours as a numeric proxy
    safe_lag_numeric = pd.to_numeric(self.df.loc[drift_idx, "processing_lag_hours"].astype(str).str.replace(",", "."), errors="coerce").fillna(0)
    self.df.loc[drift_idx, "processing_lag_hours"] = (safe_lag_numeric + noise).round(6)

    self._log(21, "Price Drift — Rounding Error", len(drift_idx))

    # -- 4.5 Negative Stock Drift 
    # stock adjustment entries entered with wrong sign
    # write offs push stock_on_hand below zero
    snapshot_rows = self.df[
        (self.df["event_type"] == "snapshot") &
        (self.df["node_id"].isin(
            df_node[df_node["tier"] == "metropolitan"]["node_id"]
        ))
    ]
    neg_idx = snapshot_rows.sample(
        n=min(10, len(snapshot_rows)), random_state=59
    ).index
    self.df.loc[neg_idx, "stock_on_hand"] = (
        pd.Series(
            [random.randint(-200, -5) for _ in range(len(neg_idx))],
            index=neg_idx
        )
    )
    self._log(22, "Negative Stock Drift", len(neg_idx))

    # -- 4.6 Seasonal Demand Masking
    # 5 nodes show lower demand during spike windows (throttled feed)
    spike_nodes = self.df["node_id"].value_counts().index[:5].tolist()
    spike_dates = pd.to_datetime([
        "2025-12-31", "2026-03-14", "2026-04-14"
    ])
    
    masking_mask = (
        self.df["node_id"].isin(spike_nodes) & 
        self.df["event_timestamp"].dt.normalize().isin(spike_dates) &
        (self.df["event_type"] == "customer_fulfillment")
    )

    matched_masking_df = self.df[masking_mask].sample(frac=0.5, random_state=62)
    self.df = self.df[~self.df.index.isin(matched_masking_df.index)]
    self._log(23, "Seasonal Demand Masking", len(matched_masking_df))
    return self

# patch the method on the class
SupplyChainChaosEngine.apply_market_chaos = apply_market_chaos
print("apply_market_chaos() patched onto ChaosEngine.")

apply_market_chaos() patched onto ChaosEngine.


### **Apply LAD (late arriving data) Chaos:**

In [24]:
def apply_lad_chaos(self):
    """
    Injects 2 late-arriving data failures.
    These corrupt event sequence rather than values.
    See chaos_library.md Category 5 for full specifications.
    """
    print("Applying late-arriving data chaos...")

    # reset the index 
    self.df = self.df.reset_index(drop=True)

    # -- 5.1 Late-Arriving Event 
    # degraded scanner buffers events locally during connectivity loss
    # uploads 2-3 days late: event_timestamp is Monday, but it arrives Wednesday
    # processing_lag_hours is set to 48-72 hours to simulate the delay
    degraded_rows = self.df[
        self.df["device_id"].isin(degraded_scanners) &
        (self.df["event_type"] == "customer_fulfillment")
    ]
    lad_idx = degraded_rows.sample(
        frac=0.03, random_state=60
    ).index

    random_lags = [round(random.uniform(48, 72), 2) for _ in range(len(lad_idx))]
    self.df.loc[lad_idx, "processing_lag_hours"] = random_lags

    self._log(24, "Late-Arriving Event (48-72h lag)", len(lad_idx))

    # -- 5.2 Out-of-Order Sequence 
    # fulfillment arrives before the inbound transfer that restocked it
    # in arrival order the fulfillment looks like it happened against zero stock
    # implemented by swapping timestamps on 10 transfer/fulfillment pairs
    # find inbound transfers that have a matching fulfillment on the same day
    transfer_rows = self.df[self.df["event_type"] == "inbound_transfer"]

    swapped = 0
    for transfer_idx, transfer in transfer_rows.sample(
        n=min(50, len(transfer_rows)), random_state=61
    ).iterrows():
        node = transfer["node_id"]
        prod = transfer["product_id"]
        t_timestamp = transfer["event_timestamp"]

        prod_variants = [prod]
        if prod == "PROD_001":
            prod_variants.append("PROD_001_OLD_SKU")
        elif prod == "PROD_001_OLD_SKU":
            prod_variants.append("PROD_001")

        match = self.df[
            (self.df["node_id"] == node) &
            (self.df["product_id"].isin(prod_variants)) &
            (self.df["event_type"] == "customer_fulfillment") &
            ((self.df["event_timestamp"] - t_timestamp).abs() <= pd.Timedelta(days=1))
        ]

        if len(match) == 0:
            continue

        fulfillment_idx = match.index[0]

        orig_transfer_ts = self.df.loc[transfer_idx, "event_timestamp"]
        orig_fulfillment_ts = self.df.loc[fulfillment_idx, "event_timestamp"]

        self.df.loc[transfer_idx, "event_timestamp"] = orig_fulfillment_ts
        self.df.loc[fulfillment_idx, "event_timestamp"] = orig_transfer_ts
        swapped += 1

        if swapped >= 10:
            break

    self._log(25, "Out-of-Order Sequence (timestamp swap)", swapped)

    return self

# patch the method on the class
SupplyChainChaosEngine.apply_lad_chaos = apply_lad_chaos
print("apply_lad_chaos() patched onto ChaosEngine.")

apply_lad_chaos() patched onto ChaosEngine.


### **Run the chaosEngine, Validate, and save**


In [25]:
# -- run the ChaosEngine:
print("=" * 42)
print("PhantomProof — Chaos Injection Engine")
print("=" * 42)
print(f"Baseline events: {len(df_events):,}\n")

engine = SupplyChainChaosEngine(df_events, df_node)
(engine.apply_temporal_chaos()
       .apply_structural_chaos()
       .apply_operational_chaos()
       .apply_market_chaos()
       .apply_lad_chaos())

# -- summary
engine.summary()

# -- validation
print("\n=== Validation ===")
df_chaos = engine.df

print(f"Total rows: {len(df_chaos):,}")
print(f"Baseline rows: {len(df_events):,}")
net = len(df_chaos) - len(df_events)
print(f"Net row delta: {net:,}")
print(f"  +150 injected (100 exact dupes + 50 sub-second dupes)")
print(f"  - 70 removed  (Node Blackout)")
print(f"  -100 removed  (Seasonal Demand Masking)")
print(f"  = {net} net — expected, not data loss")



print(f"Ghost inventory rows (fulfillment only): "
      f"{((df_chaos['event_type'] == 'customer_fulfillment') & 
          (pd.to_numeric(df_chaos['units_sold'], errors='coerce') > 
           df_chaos['stock_on_hand'])).sum()}")

print(f"Negative stock rows: "
      f"{(df_chaos['stock_on_hand'] < 0).sum()}")

print(f"Null RTO reasons: "
      f"{((df_chaos['event_type'] == 'rto_return') & (df_chaos['rto_reason'].isna())).sum()}")

print(f"Schema poisoned rows: "
      f"{df_chaos['units_sold'].astype(str).str.contains(' units').sum()}")

print(f"Future timestamps: "
      f"{(df_chaos['event_timestamp'] > pd.Timestamp('2026-06-30')).sum()}")

print(f"Extreme lag rows (>24h): "
      f"{(pd.to_numeric(df_chaos['processing_lag_hours'], errors='coerce') > 24).sum()}")

print(f"SKU migration rows: "
      f"{(df_chaos['product_id'] == 'PROD_001_OLD_SKU').sum()}")

print(f"Exact duplicate rows: "
      f"{df_chaos.duplicated(subset=['node_id','product_id','units_sold','event_timestamp']).sum()}")

# ── save ─────────────────────────────────────────────────────────────────────
engine.save(
    events_path="events_chaotic.csv",
    nodes_path="nodes_chaotic.csv"
)

print("\nNB3 complete. Chaotic dataset ready for Bronze ingestion.")

PhantomProof — Chaos Injection Engine
Baseline events: 436,143

Applying temporal chaos...
 [01] UTC/IST Drift                             85360 rows
 [02] Timezone Double-Conversion                 3481 rows
 [03] Batch Heartbeat (3 dates)                  1393 rows
 [04] Future Timestamp                           4318 rows
 [05] Midnight Rollover                         10795 rows
 [06] Processing Lag Smear                       2597 rows
Applying structural chaos...
 [07] Schema Poisoning — Unit Suffix               50 rows
 [08] Schema Poisoning: Decimal Separator         895 rows
 [09] Address and City Corruption                   3 rows
 [10] Null RTO Reason                              20 rows
 [11] UoM Mismatch — Cases vs Units                 5 rows
Applying operational chaos...
 [12] Ghost Inventory                              20 rows
 [13] Duplicate Echoes — Exact                    100 rows
 [14] Duplicate Echoes — Sub-second                50 rows
 [15] Node Blackout (DS_